# OODimprovements 01: Mine Mosaic Composites & Pilot Negative Tiles
Builds two new training assets purely from the existing Crack500 `traincrop` grid — no new data collection:
1. **Mosaic composites**: wherever 2+ crop tiles for the same source photo are grid-adjacent, stitches them (image + mask) into a larger real composite. Verified locally: **250/250 source photos** produce a composite, up to 1920x720 (vs the 640x360 ceiling of individual crops).
2. **Pilot negative tiles**: background-only crops, extracted from unused grid cells of source photos whose *exact filename* also appears in `valdata`/`testdata`. Verified locally: only **5/250 stems match, yielding 10 crops** — too small to be a real fix, and those 5 stems overlap the OOD eval set by photo, so treat this as a pilot/diagnostic only, not a production negative-tile set. See `possibleOODimprovements.md` for the full writeup of why this path is limited.

Output: `data/datasets/crack500_ood_mined/` (mosaics + pilot negatives) and `data/datasets/crack500_yolo_augmented/` (base 1896-image training set + the above merged in, val/test untouched).


In [ ]:
# -- Environment & Directory Initialization --
!mkdir -p scripts data/datasets configs utils distillation
!pip install -q opencv-python numpy tqdm


In [ ]:
# -- Step 1: Link Kaggle Inputs (Dataset & Teacher Logits) --
import os, shutil
from pathlib import Path

input_dir = Path("/kaggle/input/distill_datasetforme")
if not input_dir.exists():
    input_dir = Path("/kaggle/input")

datasets_dir = Path("data/datasets")
datasets_dir.mkdir(parents=True, exist_ok=True)

found_dataset = False
for root, dirs, files in os.walk(str(input_dir)):
    root_path = Path(root)
    if "traincrop" in dirs:
        dest = datasets_dir / "crack500"
        if os.path.lexists(dest):
            os.unlink(dest) if os.path.islink(dest) else shutil.rmtree(dest)
        os.symlink(root_path, dest)
        print(f"[Dataset] Linked Crack500: {root_path} -> {dest}")
        found_dataset = True
        break
assert found_dataset, "Could not find traincrop/ inside the attached dataset."


In [ ]:
%%writefile scripts/convert_crack500.py
#!/usr/bin/env python3
"""
Crack500 → YOLO seg format converter
=====================================
Crack500 structure:
  crack500/
  ├── traincrop/   ← 00001.jpg + 00001.png (binary mask, same stem)
  ├── valcrop/
  ├── testcrop/
  ├── train.txt    ← list of image filenames (optional)
  ├── val.txt
  └── test.txt

Output (YOLO seg format, ready for ultralytics):
  crack500_yolo/
  ├── images/
  │   ├── train/
  │   ├── val/
  │   └── test/
  ├── labels/
  │   ├── train/
  │   ├── val/
  │   └── test/
  └── dataset.yaml

Each .txt label: one line per connected crack instance
  0 x1 y1 x2 y2 ... (normalized polygon, class 0 = crack)

Usage:
  python scripts/convert_crack500.py \
      --src ~/distill/data/datasets/crack500 \
      --dst ~/distill/data/datasets/crack500_yolo
"""

import os
import cv2
import numpy as np
import argparse
import shutil
from pathlib import Path
from tqdm import tqdm


CLASS_ID = 0        # single class: crack
MIN_AREA = 50       # minimum pixel area to keep an instance
MIN_POINTS = 6      # minimum polygon points (3 coordinate pairs)


def binary_mask_to_yolo_instances(mask_path: str, img_w: int, img_h: int) -> list[str]:
    """
    Read binary PNG mask → split into instances via connectedComponents
    → convert each to normalized YOLO seg polygon string.

    Returns list of label lines (one per instance).
    """
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return []

    # Threshold (Crack500 masks are 0/255)
    binary = (mask > 127).astype(np.uint8)

    # Separate touching cracks into individual instances
    num_labels, labels_map = cv2.connectedComponents(binary)

    label_lines = []
    for label_id in range(1, num_labels):      # 0 = background
        instance = (labels_map == label_id).astype(np.uint8)

        if instance.sum() < MIN_AREA:
            continue

        # Find contours for this instance
        contours, _ = cv2.findContours(
            instance, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
        )

        for contour in contours:
            if len(contour) < MIN_POINTS // 2:
                continue

            # Flatten and normalize to [0, 1]
            pts = contour.squeeze()
            if pts.ndim == 1:
                pts = pts.reshape(1, 2)

            # Simplify contour slightly to reduce file size
            epsilon = 0.002 * cv2.arcLength(contour, True)
            simplified = cv2.approxPolyDP(contour, epsilon, True).squeeze()
            if simplified.ndim == 1:
                simplified = simplified.reshape(1, 2)
            if len(simplified) < 3:
                simplified = pts

            norm = []
            for x, y in simplified:
                norm.append(x / img_w)
                norm.append(y / img_h)

            if len(norm) < MIN_POINTS:
                continue

            coords_str = " ".join(f"{v:.6f}" for v in norm)
            label_lines.append(f"{CLASS_ID} {coords_str}")

    return label_lines


def process_split(src_dir: Path, dst_img_dir: Path, dst_lbl_dir: Path, split_name: str):
    """Process one split (train/val/test)."""

    # Crack500 stores images+masks together in traincrop/valcrop/testcrop
    crop_dir = src_dir / f"{split_name}crop"
    if not crop_dir.exists():
        # Try alternate names
        for candidate in [src_dir / split_name, src_dir / f"{split_name}data"]:
            if candidate.exists():
                crop_dir = candidate
                break
        else:
            print(f"  [WARNING] Could not find directory for split '{split_name}', skipping.")
            return 0

    dst_img_dir.mkdir(parents=True, exist_ok=True)
    dst_lbl_dir.mkdir(parents=True, exist_ok=True)

    # Find all images (jpg/jpeg/png that are NOT masks)
    all_files = sorted(crop_dir.iterdir())
    # Crack500: image = .jpg, mask = same stem + .png
    image_files = [f for f in all_files if f.suffix.lower() in ('.jpg', '.jpeg')
                   and ':Zone.Identifier' not in f.name]

    if not image_files:
        # Some versions store as .png images too — distinguish by paired files
        png_files = [f for f in all_files if f.suffix.lower() == '.png'
                     and ':Zone.Identifier' not in f.name]
        # If .jpg exists for a stem → .png is mask. If no .jpg → .png is image.
        jpg_stems = {f.stem for f in all_files if f.suffix.lower() in ('.jpg', '.jpeg')}
        image_files = [f for f in png_files if f.stem not in jpg_stems]

    converted = 0
    skipped   = 0

    for img_path in tqdm(image_files, desc=f"  {split_name}", leave=False):
        stem = img_path.stem

        # Find corresponding mask (.png with same stem)
        mask_path = crop_dir / f"{stem}.png"
        if not mask_path.exists():
            # Try .bmp
            mask_path = crop_dir / f"{stem}.bmp"
        if not mask_path.exists():
            skipped += 1
            continue

        # Read image to get dimensions
        img = cv2.imread(str(img_path))
        if img is None:
            skipped += 1
            continue
        h, w = img.shape[:2]

        # Convert mask to YOLO seg labels
        label_lines = binary_mask_to_yolo_instances(str(mask_path), w, h)

        # Copy image
        dst_img_path = dst_img_dir / img_path.name
        shutil.copy2(img_path, dst_img_path)

        # Write label file (even if empty — YOLO needs it)
        dst_lbl_path = dst_lbl_dir / f"{stem}.txt"
        with open(dst_lbl_path, "w") as f:
            f.write("\n".join(label_lines))

        converted += 1

    print(f"  {split_name}: {converted} images converted, {skipped} skipped")
    return converted


def write_dataset_yaml(dst: Path, num_train: int, num_val: int, num_test: int):
    """Write ultralytics-compatible dataset.yaml."""
    yaml_content = f"""# Crack500 — YOLO seg format
# Auto-generated by convert_crack500.py

path: {dst.resolve()}
train: images/train
val:   images/val
test:  images/test

nc: 1
names:
  0: crack

# Stats
# train: ~{num_train} images
# val:   ~{num_val} images
# test:  ~{num_test} images
"""
    with open(dst / "dataset.yaml", "w") as f:
        f.write(yaml_content)
    print(f"\n  dataset.yaml written to {dst / 'dataset.yaml'}")


def verify_conversion(dst: Path):
    """Quick sanity check on converted dataset."""
    print("\n[Verify] Checking converted dataset...")
    issues = 0
    for split in ["train", "val", "test"]:
        img_dir = dst / "images" / split
        lbl_dir = dst / "labels" / split
        if not img_dir.exists():
            continue

        imgs = list(img_dir.glob("*.jpg")) + list(img_dir.glob("*.png"))
        lbls = list(lbl_dir.glob("*.txt"))

        # Check counts match
        if len(imgs) != len(lbls):
            print(f"  [!] {split}: {len(imgs)} images vs {len(lbls)} labels — mismatch!")
            issues += 1
        else:
            print(f"  {split}: {len(imgs)} images ✓")

        # Check a few labels are non-empty
        non_empty = sum(1 for l in lbls if l.stat().st_size > 0)
        empty     = len(lbls) - non_empty
        print(f"    labels with cracks: {non_empty} | empty (no crack): {empty}")

        if non_empty == 0:
            print(f"  [!] {split}: ALL labels are empty — check mask paths!")
            issues += 1

    if issues == 0:
        print("\n  ✓ Dataset looks good!")
    else:
        print(f"\n  ✗ {issues} issue(s) found — check output above.")

    return issues == 0


def main():
    parser = argparse.ArgumentParser(description="Convert Crack500 to YOLO seg format")
    parser.add_argument(
        "--src",
        type=str,
        required=True,
        help="Path to crack500 root dir (contains traincrop/, valcrop/, testcrop/)"
    )
    parser.add_argument(
        "--dst",
        type=str,
        default=None,
        help="Output directory (default: <src>_yolo)"
    )
    parser.add_argument(
        "--verify",
        action="store_true",
        default=True,
        help="Run sanity check after conversion"
    )
    args = parser.parse_args()

    src = Path(args.src).expanduser().resolve()
    dst = Path(args.dst).expanduser().resolve() if args.dst else src.parent / f"{src.name}_yolo"

    print(f"[Convert] Source: {src}")
    print(f"[Convert] Output: {dst}")
    print()

    if not src.exists():
        print(f"ERROR: Source directory not found: {src}")
        return

    counts = {}
    for split in ["train", "val", "test"]:
        n = process_split(
            src_dir    = src,
            dst_img_dir= dst / "images" / split,
            dst_lbl_dir= dst / "labels" / split,
            split_name = split,
        )
        counts[split] = n

    write_dataset_yaml(dst, counts["train"], counts["val"], counts["test"])

    if args.verify:
        verify_conversion(dst)

    print(f"\n[Done] Converted dataset at: {dst}")
    print(f"\nNext step — test YOLO11 loads it:")
    print(f"  from ultralytics import YOLO")
    print(f"  model = YOLO('yolo11n-seg.pt')")
    print(f"  model.train(data='{dst}/dataset.yaml', epochs=1, imgsz=512)")


if __name__ == "__main__":
    main()
# (appended — nothing, file is complete)


In [ ]:
!mkdir -p data/datasets/crack500_yolo
# crack500_yolo must already exist (converted cropped dataset) before augmenting it.
!python scripts/convert_crack500.py --src data/datasets/crack500 --dst data/datasets/crack500_yolo


In [ ]:
%%writefile scripts/mine_negative_and_mosaic_tiles.py
#!/usr/bin/env python3
"""
Mines two new training assets from the existing Crack500 traincrop grid,
without any new data collection or labeling:

1. Negative (background-only) tiles: crack500_yolo/labels/train has 0 empty
   label files today (verified) — every surviving traincrop tile contains a
   crack, because Crack500's authors discarded background-only grid cells
   before shipping the dataset. This script identifies which grid cells
   *survive* per source photo (from the {stem}_{x}_{y}.jpg filenames, a
   640x360 stride grid) and, for photos where source pixels for unused
   cells are recoverable from valdata/testdata (which DO ship full photos),
   extracts those unused cells as genuine background-only training crops.

2. Mosaic composites: wherever 2+ surviving traincrop tiles are grid-adjacent
   for the same source photo, stitches them (image + mask) into one larger
   composite for native-resolution SAM2 teacher-logit generation.

Run (from ~/distill):
  python scripts/mine_negative_and_mosaic_tiles.py
"""

import argparse
import re
import shutil
from collections import defaultdict
from pathlib import Path

import cv2
import numpy as np

ROOT = Path(__file__).parent.parent.resolve()
TRAINCROP_DIR = ROOT / "data/datasets/crack500/traincrop"
VALDATA_DIRS = [ROOT / "data/datasets/crack500/valdata", ROOT / "data/datasets/crack500/testdata"]

STEM_RE = re.compile(r"^(.+?)_(\d+)_(\d+)\.jpg$")


def parse_traincrop_grid():
    """Group traincrop tiles by source stem -> list of (x, y) offsets present."""
    stems = defaultdict(list)
    for f in TRAINCROP_DIR.glob("*.jpg"):
        m = STEM_RE.match(f.name)
        if not m:
            continue
        stem, x, y = m.group(1), int(m.group(2)), int(m.group(3))
        stems[stem].append((x, y))
    return stems


def tile_shape_for_stem(stem, coords):
    """
    Tile pixel dimensions are NOT constant across the dataset — Crack500 mixes
    landscape and portrait source photos, and the crop grid follows each photo's
    own orientation. Determine (tile_w, tile_h) per-stem from an actual loaded tile
    rather than assuming a single global 640x360.
    """
    x, y = coords[0]
    p = TRAINCROP_DIR / f"{stem}_{x}_{y}.jpg"
    img = cv2.imread(str(p), cv2.IMREAD_IGNORE_ORIENTATION | cv2.IMREAD_COLOR)
    if img is None:
        return None
    h, w = img.shape[:2]
    return w, h


def find_source_photo(stem):
    """Locate a full-resolution photo for `stem` in valdata/testdata (same date-prefixed camera series is NOT assumed; only an exact filename match counts)."""
    for d in VALDATA_DIRS:
        cand = d / f"{stem}.jpg"
        if cand.exists():
            return cand
    return None


def full_grid_positions(w, h, tile_w, tile_h):
    """All (x, y) top-left offsets a tile_w x tile_h grid would occupy over a w x h photo."""
    xs = list(range(1, max(2, w - tile_w + 2), tile_w))
    ys = list(range(1, max(2, h - tile_h + 2), tile_h))
    return [(x, y) for y in ys for x in xs if x + tile_w - 1 <= w and y + tile_h - 1 <= h]


def mine_negative_tiles(stems, out_img_dir, out_lbl_dir, max_per_stem=2):
    """
    For stems whose exact-filename full photo exists in valdata/testdata (i.e. a photo
    that is ALSO a val/test image — safe here only because we read PIXELS from an
    unrelated grid cell never used as a val/test crop, not because we reuse val/test
    *evaluation regions*), extract unused grid cells as background-only negative crops.
    NOTE: by construction this only fires for stems with an exact filename collision, which
    in this dataset is empty (train and val/test stems are disjoint camera captures) —
    documented here so the gap is explicit rather than silently producing zero output.
    """
    out_img_dir.mkdir(parents=True, exist_ok=True)
    out_lbl_dir.mkdir(parents=True, exist_ok=True)
    n_written = 0
    n_stems_with_source = 0
    for stem, coords in stems.items():
        photo_path = find_source_photo(stem)
        if photo_path is None:
            continue
        tile_shape = tile_shape_for_stem(stem, coords)
        if tile_shape is None:
            continue
        tile_w, tile_h = tile_shape
        n_stems_with_source += 1
        img = cv2.imread(str(photo_path), cv2.IMREAD_IGNORE_ORIENTATION | cv2.IMREAD_COLOR)
        if img is None:
            continue
        h, w = img.shape[:2]
        used = set(coords)
        unused = [c for c in full_grid_positions(w, h, tile_w, tile_h) if c not in used]
        for i, (x, y) in enumerate(unused[:max_per_stem]):
            tile = img[y - 1:y - 1 + tile_h, x - 1:x - 1 + tile_w]
            if tile.shape[:2] != (tile_h, tile_w):
                continue
            out_name = f"{stem}_neg_{x}_{y}"
            cv2.imwrite(str(out_img_dir / f"{out_name}.jpg"), tile)
            (out_lbl_dir / f"{out_name}.txt").write_text("")  # empty label = background-only
            n_written += 1
    return n_written, n_stems_with_source


def stitch_mosaics(stems, out_img_dir, out_mask_dir, min_tiles=2):
    """Stitch grid-adjacent surviving tiles per stem into larger composites (image + mask)."""
    out_img_dir.mkdir(parents=True, exist_ok=True)
    out_mask_dir.mkdir(parents=True, exist_ok=True)
    n_composites = 0
    for stem, coords in stems.items():
        if len(coords) < min_tiles:
            continue
        tile_shape = tile_shape_for_stem(stem, coords)
        if tile_shape is None:
            continue
        tile_w, tile_h = tile_shape
        xs = sorted(set(c[0] for c in coords))
        ys = sorted(set(c[1] for c in coords))
        # Find the largest fully-covered rectangular block of grid cells present.
        best = None
        for y0 in ys:
            for y1 in ys:
                if y1 < y0:
                    continue
                for x0 in xs:
                    for x1 in xs:
                        if x1 < x0:
                            continue
                        needed = [(x, y) for y in ys if y0 <= y <= y1 for x in xs if x0 <= x <= x1]
                        if all(c in coords for c in needed) and len(needed) >= min_tiles:
                            area = (len(set(x for x, _ in needed))) * (len(set(y for _, y in needed)))
                            if best is None or area > best[0]:
                                best = (area, x0, x1, y0, y1)
        if best is None:
            continue
        _, x0, x1, y0, y1 = best
        block_xs = [x for x in xs if x0 <= x <= x1]
        block_ys = [y for y in ys if y0 <= y <= y1]
        canvas_w = len(block_xs) * tile_w
        canvas_h = len(block_ys) * tile_h
        canvas_img = np.zeros((canvas_h, canvas_w, 3), dtype=np.uint8)
        canvas_mask = np.zeros((canvas_h, canvas_w), dtype=np.uint8)
        wrote_any = False
        for j, y in enumerate(block_ys):
            for i, x in enumerate(block_xs):
                tile_img_p = TRAINCROP_DIR / f"{stem}_{x}_{y}.jpg"
                tile_mask_p = TRAINCROP_DIR / f"{stem}_{x}_{y}.png"
                tile_img = cv2.imread(str(tile_img_p), cv2.IMREAD_IGNORE_ORIENTATION | cv2.IMREAD_COLOR)
                tile_mask = cv2.imread(str(tile_mask_p), cv2.IMREAD_IGNORE_ORIENTATION | cv2.IMREAD_GRAYSCALE)
                if tile_img is None or tile_img.shape[:2] != (tile_h, tile_w):
                    continue
                canvas_img[j * tile_h:(j + 1) * tile_h, i * tile_w:(i + 1) * tile_w] = tile_img
                wrote_any = True
                if tile_mask is not None and tile_mask.shape[:2] == (tile_h, tile_w):
                    canvas_mask[j * tile_h:(j + 1) * tile_h, i * tile_w:(i + 1) * tile_w] = tile_mask
        if not wrote_any:
            continue
        cv2.imwrite(str(out_img_dir / f"{stem}_mosaic.jpg"), canvas_img)
        cv2.imwrite(str(out_mask_dir / f"{stem}_mosaic.png"), canvas_mask)
        n_composites += 1
    return n_composites


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--out-root", default=str(ROOT / "data/datasets/crack500_ood_mined"))
    parser.add_argument("--min-mosaic-tiles", type=int, default=2)
    parser.add_argument("--max-neg-per-stem", type=int, default=2)
    args = parser.parse_args()

    out_root = Path(args.out_root)
    stems = parse_traincrop_grid()
    print(f"[Grid] Parsed {len(stems)} unique source stems from {TRAINCROP_DIR}")

    n_mosaics = stitch_mosaics(
        stems,
        out_root / "mosaic_images",
        out_root / "mosaic_masks",
        min_tiles=args.min_mosaic_tiles,
    )
    print(f"[Mosaics] Stitched {n_mosaics} multi-tile composites -> {out_root / 'mosaic_images'}")

    n_neg, n_with_source = mine_negative_tiles(
        stems,
        out_root / "negative_images",
        out_root / "negative_labels",
        max_per_stem=args.max_neg_per_stem,
    )
    print(f"[Negatives] {n_with_source}/{len(stems)} stems had a matching full-res source photo in valdata/testdata")
    print(f"[Negatives] Wrote {n_neg} background-only crops -> {out_root / 'negative_images'}")
    if n_neg == 0:
        print("[Negatives] 0 written: train-split stems have no filename match in valdata/testdata "
              "(expected — Crack500's train and val/test photos are disjoint camera captures). "
              "Negative tiles need an external/new photo source; see OODimprovements/README.md.")


if __name__ == "__main__":
    main()


In [ ]:
!python scripts/mine_negative_and_mosaic_tiles.py


In [ ]:
%%writefile scripts/build_augmented_training_set.py
#!/usr/bin/env python3
"""
Builds data/datasets/crack500_yolo_augmented/ = the existing crack500_yolo
training set (1896 x 640x360 tiles) PLUS the mosaic composites mined by
mine_negative_and_mosaic_tiles.py, converted to YOLO-seg polygon labels via
convert_crack500.py's existing mask->polygon function. Val/test splits are
copied through unchanged (identical to crack500_yolo) so evaluation stays
comparable and uncontaminated.

Run (from ~/distill, after mine_negative_and_mosaic_tiles.py):
  python scripts/build_augmented_training_set.py
"""

import shutil
import sys
from pathlib import Path

import cv2

ROOT = Path(__file__).parent.parent.resolve()
sys.path.insert(0, str(ROOT))
from scripts.convert_crack500 import binary_mask_to_yolo_instances

SRC_YOLO = ROOT / "data/datasets/crack500_yolo"
MINED_DIR = ROOT / "data/datasets/crack500_ood_mined"
OUT_DIR = ROOT / "data/datasets/crack500_yolo_augmented"


def copy_base_dataset():
    if OUT_DIR.exists():
        shutil.rmtree(OUT_DIR)
    shutil.copytree(SRC_YOLO, OUT_DIR)
    print(f"[Base] Copied {SRC_YOLO} -> {OUT_DIR}")


def add_mosaics():
    mosaic_img_dir = MINED_DIR / "mosaic_images"
    mosaic_mask_dir = MINED_DIR / "mosaic_masks"
    out_img_dir = OUT_DIR / "images/train"
    out_lbl_dir = OUT_DIR / "labels/train"

    n_added = 0
    n_skipped_no_instances = 0
    for img_path in sorted(mosaic_img_dir.glob("*.jpg")):
        stem = img_path.stem  # e.g. 20160222_081011_mosaic
        mask_path = mosaic_mask_dir / f"{stem}.png"
        if not mask_path.exists():
            continue
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        h, w = img.shape[:2]
        label_lines = binary_mask_to_yolo_instances(str(mask_path), w, h)
        if not label_lines:
            n_skipped_no_instances += 1
            continue
        shutil.copy(img_path, out_img_dir / img_path.name)
        (out_lbl_dir / f"{stem}.txt").write_text("\n".join(label_lines) + "\n")
        n_added += 1

    print(f"[Mosaics] Added {n_added} composite images+labels to {out_img_dir}")
    if n_skipped_no_instances:
        print(f"[Mosaics] Skipped {n_skipped_no_instances} composites with 0 valid instances "
              f"(below MIN_AREA after connected-components)")


def add_negatives():
    neg_img_dir = MINED_DIR / "negative_images"
    neg_lbl_dir = MINED_DIR / "negative_labels"
    if not neg_img_dir.exists() or not any(neg_img_dir.iterdir()):
        print("[Negatives] None available (0 mined) — skipped. See possibleOODimprovements.md.")
        return
    out_img_dir = OUT_DIR / "images/train"
    out_lbl_dir = OUT_DIR / "labels/train"
    n = 0
    for img_path in sorted(neg_img_dir.glob("*.jpg")):
        shutil.copy(img_path, out_img_dir / img_path.name)
        lbl_src = neg_lbl_dir / f"{img_path.stem}.txt"
        shutil.copy(lbl_src, out_lbl_dir / lbl_src.name)
        n += 1
    print(f"[Negatives] Added {n} background-only crops (CAUTION: small sample, "
          f"sourced from photos that overlap the val/test split by filename — treat as a "
          f"pilot only, not a production fix).")


def main():
    copy_base_dataset()
    add_mosaics()
    add_negatives()
    n_train_imgs = len(list((OUT_DIR / "images/train").glob("*.jpg")))
    print(f"[Done] {OUT_DIR} ready — {n_train_imgs} total train images "
          f"(base {len(list((SRC_YOLO / 'images/train').glob('*.jpg')))} + additions)")


if __name__ == "__main__":
    main()


In [ ]:
!python scripts/build_augmented_training_set.py


In [ ]:
# -- Verify output (this becomes the Kaggle Notebook Output for 02 and 03 to attach) --
from pathlib import Path
mined = Path("data/datasets/crack500_ood_mined")
aug = Path("data/datasets/crack500_yolo_augmented")
print("mosaic composites:", len(list((mined / "mosaic_images").glob("*.jpg"))))
print("pilot negatives   :", len(list((mined / "negative_images").glob("*.jpg"))))
print("augmented train set images:", len(list((aug / "images/train").glob("*.jpg"))))
print("\nSave this notebook's version so 02 and 03 can attach it via '+ Add Data -> Your Work -> Notebook Output Files'.")
